# RAG Meets Graph Analytics: Building Hybrid Retrieval on Your Existing Iceberg Data

## How to combine vector search, BM25, and graph traversal in [BenoStreamDB](https://github.com/benolabsai/benostreamdb) — without operating a separate vector database, a search engine, and a graph service

> **TL;DR:** BenoStreamDB brings dense vector search (HNSW), full-text keyword retrieval (BM25), and graph traversal (CSR) into a single in-process engine directly over Apache Iceberg lakehouse tables. You don't need to coordinate Pinecone, Elasticsearch, and Neo4j just to run Graph RAG. Code, docs, and benchmarks: [github.com/benolabsai/benostreamdb](https://github.com/benolabsai/benostreamdb).

---

The idea that retrieval and structured knowledge are complements, not competitors is well established. Retrieval-Augmented Generation grounds a model in retrieved passages ([Lewis et al., 2020](https://papers.nips.cc/paper/2020/hash/6b493230205f780e1bc26945df7481e5-Abstract.html)); knowledge graphs give it explicit, traversable relationships ([Hogan et al., 2021](https://doi.org/10.1145/3447772)). Recent work argues the two should be unified rather than run as separate stacks ([Pan et al., 2024](https://doi.org/10.1109/TKDE.2024.3352100); [Gao et al., 2023](https://arxiv.org/abs/2312.10997)), and Microsoft's GraphRAG demonstrates what graph-structured retrieval buys you at corpus scale ([Edge et al., 2024](https://arxiv.org/abs/2404.16130); [Gutiérrez et al., 2024](https://arxiv.org/abs/2405.14831)).

The thesis for AI agents is straightforward: **don't choose between RAG and knowledge frameworks — use both.** Semantic search over vectors for fuzzy conceptual understanding; structured graph reasoning for explicit multi-hop relationships.

### The Three-System Tax

In production, the supporting infrastructure is frequently fragmented. Most teams end up deploying **three separate systems**:
1. A dedicated vector database (e.g., Pinecone or Qdrant) for embeddings,
2. A search engine (e.g., Elasticsearch or OpenSearch) for keyword matching,
3. A graph database (e.g., Neo4j) for relationship traversal.

Each system demands its own cluster, operational runbook, synchronization pipeline, and authentication boundary. Documents get duplicated across three datastores. Re-embedding passages forces dual-write coordination; schema changes cascade across three distinct APIs; and every agent retrieval loop pays a multi-hop network latency tax.

**What if retrieval, search, and graph analytics lived in one database?**

That is what BenoStreamDB was built for. In this walkthrough, we build an end-to-end Graph RAG system over a real corpus:
- Ingest documents and chunk them *within* section boundaries.
- Embed chunks with a production model (`BAAI/bge-large-en-v1.5`, 1024-d).
- Layer secondary vector (HNSW) and text (BM25) indexes directly onto the Iceberg table.
- Execute hybrid retrieval with server-side Reciprocal Rank Fusion (RRF).
- Evaluate retrieval quality quantitatively against labeled ground truth from the [BEIR SciFact](https://github.com/allenai/scifact) benchmark.
- Derive a section-level similarity graph using GPU-accelerated pairwise distance kernels (`bsdb.cdist`).
- Traverse graph neighbors and shortest paths with in-process SQL table functions.
- Run Graph RAG (local seed discovery + subgraph expansion + Personalized PageRank) and generate verified answers with live LLM citations.
- Perform Louvain community detection to summarize global corpus themes.

**Who this is for:** Engineers and data architects building retrieval-augmented agents who want unified lakehouse storage without stitching together three disconnected services.

**Developer preview status:** This walkthrough reflects the **Developer Preview (v0.13.1)** of [BenoStreamDB](https://github.com/benolabsai/benostreamdb). Features ready today include the embedded engine and Flight SQL server, unified Parquet/Iceberg storage, synchronous and backgrounded index builds (HNSW, BM25, and CSR graphs), hybrid/graph retrieval algorithms, GPU batch distance kernels (`bsdb.cdist`), and dynamic heap discipline with proactive cache eviction. What is intentionally *not* in this preview is distributed multi-node consensus or cross-table multi-statement transactions.

> **Run this tutorial yourself:** [download the notebook (.ipynb)](https://raw.githubusercontent.com/benolabsai/benostreamdb/main/blog/articles/2026-10-22_graph_rag_howto.ipynb) · [view on GitHub](https://github.com/benolabsai/benostreamdb/blob/main/blog/articles/2026-10-22_graph_rag_howto.ipynb).


## The Document Table (Plus an Edge Table)

The retrieval architecture uses **one primary document table** — one row per chunk, holding the document text, the embedding vector, and the positional metadata to stitch context together:

```sql
-- One row per chunk. Hybrid search and windowing run against this table.
CREATE TABLE doc_chunks (
    doc_id        BIGINT,
    section       TEXT,
    section_id    TEXT,          -- "doc_id:section", a stable section key
    section_node  BIGINT,        -- dense UInt64 node id for the section
    chunk_idx     INT,
    text          TEXT,
    embedding     FLOAT[1024]
) TBLPROPERTIES (
    'benostream.embedding.model' = 'BAAI/bge-large-en-v1.5',
    'benostream.embedding.dim' = '1024'
);
```

Two design choices drive this layout:
1. **Text and vectors live on the same row of the same table.** Retrieval never joins an external vector store to an external document store. When data is written, updated, or compacted, the text, metadata, and embeddings snapshot together.
2. **The chunk, not the document, is the unit of retrieval.** Embeddings are most accurate over focused passages. Storing `(doc_id, chunk_idx)` allows dynamic window expansion over adjacent chunks at query time. We track `section` so retrieval can filter by document section, and `section_node` so rows join directly to graph topology.

Because the underlying files are standard Parquet with an Apache Iceberg metadata layer, data files remain accessible to Spark, Trino, and DuckDB. Secondary indexes (HNSW graphs, BM25 postings, and CSR graphs) are stored as compact, reconstructible sidecars alongside the table.

Many practitioners avoid Apache Iceberg because traditional setups feel heavyweight: deploying Apache Spark is complete overkill for local or single-node data engineering, coordinating external catalog services (such as Nessie, Polaris, or Hive metastores) adds operational drag, and lightweight Python libraries often lack high-performance write and indexing capabilities. BenoStreamDB eliminates that barrier: an Iceberg table is created with a single in-process Python call (`bsdb.Table.create("file:///...")`), bringing open lakehouse snapshotting, time travel, and cross-engine interoperability to local workflows without infrastructure friction.

We use **SciFact** (5,183 scientific abstracts) as our corpus, and later derive a second table — **`doc_links`** — to store the graph edges.

### 1. Load the Corpus

SciFact records contain `doc_id`, `title`, `abstract` (as a list of sentences), and a `structured` flag. Keeping sentence boundaries lets us extract section labels deterministically with no secondary model.

In [ ]:
import json
import os
import tarfile
import tempfile
import urllib.request
from pathlib import Path

# Portable working directory. Override with BENOLABS_WORKDIR if desired.
# Defaults to ./benolabs_howto; if that location is not writable (e.g. a
# downloaded notebook opened from a read-only folder), fall back to the OS
# temp directory, which exists on Linux, macOS, and Windows.
def _default_workdir() -> Path:
    env = os.environ.get("BENOLABS_WORKDIR")
    if env:
        return Path(env).expanduser()
    candidate = Path.cwd() / "benolabs_howto"
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        probe = candidate / ".write-test"
        probe.write_text("", encoding="utf-8")
        probe.unlink()
        return candidate
    except OSError:
        return Path(tempfile.gettempdir()) / "benolabs_howto"


WORK_DIR = _default_workdir()
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = WORK_DIR / "data"
CORPUS_PATH = DATA_DIR / "corpus.jsonl"
ARCHIVE_PATH = DATA_DIR / "scifact_data.tar.gz"
URL = "https://scifact.s3.us-west-2.amazonaws.com/release/latest/data.tar.gz"

# Auto-download and extract SciFact corpus on first run
if not CORPUS_PATH.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE_PATH.exists():
        print(f"Downloading SciFact corpus from {URL}...")
        urllib.request.urlretrieve(URL, ARCHIVE_PATH)
    print("Extracting dataset archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        for member in tar.getmembers():
            if member.name.startswith("data/"):
                member.name = member.name[len("data/"):]
                if member.name:
                    tar.extract(member, path=DATA_DIR)
    print(f"Corpus ready at {CORPUS_PATH}")

docs = []
with CORPUS_PATH.open() as f:
    for line in f:
        r = json.loads(line)
        docs.append({
            "doc_id": int(r["doc_id"]),
            "title": r["title"],
            "sentences": r["abstract"],          # keep sentence boundaries
            "structured": bool(r.get("structured", False)),
        })

title_by_id = {d["doc_id"]: d["title"] for d in docs}

print(f"Loaded {len(docs):,} documents")
print(f"Structured (labelled sections): {sum(d['structured'] for d in docs):,}")
d = docs[0]
print(f"\nExample doc_id={d['doc_id']}")
print(f"  title: {d['title'][:80]}")
print(f"  first sentence: {d['sentences'][0][:100]}...")

### 2. Chunk *Within* Section Boundaries

A chunk should never straddle section boundaries — combining a `METHODS` sentence with a `RESULTS` sentence degrades prompt coherence. 

Each chunk captures:
- **`section`**: The labeled section header (`BACKGROUND`, `METHODS`, `RESULTS`, `CONCLUSIONS`, or `ABSTRACT` when unlabelled).
- **`chunk_idx`**: Sequential position within the document for context windowing.
- **`section_id`**: A stable section key (`f"{doc_id}:{section}"`).
- **`section_node`**: A dense UInt64 node identifier for graph operations.

We pack sentences into ~80-word chunks — short enough for focused vector embeddings, long enough for BM25 term weighting.

In [ ]:
import re
from collections import defaultdict

SECTION_RE = re.compile(
    r"^(BACKGROUND|OBJECTIVE|OBJECTIVES|PURPOSE|INTRODUCTION|IMPORTANCE|"
    r"METHODS?|DESIGN|SETTING|PARTICIPANTS|INTERVENTION|MEASUREMENTS|"
    r"RESULTS?|FINDINGS|DISCUSSION|CONCLUSIONS?|SIGNIFICANCE)\b"
)


def labelled(sentences):
    """Yield (section, sentence). Detects all-caps section headers."""
    section = "ABSTRACT"
    for s in sentences:
        m = SECTION_RE.match(s)
        if m:
            section = m.group(1)
            s = s[m.end():].strip()
        if s:
            yield section, s


def chunk_doc(doc, max_words=80):
    """List of (section, text), never crossing a section boundary."""
    out, buf, n, cur = [], [], 0, None
    for section, s in labelled(doc["sentences"]):
        if cur is not None and section != cur and buf:
            out.append((cur, " ".join(buf))); buf, n = [], 0
        cur = section
        w = len(s.split())
        if buf and n + w > max_words:
            out.append((cur, " ".join(buf))); buf, n = [], 0
        buf.append(s); n += w
    if buf:
        out.append((cur, " ".join(buf)))
    return out


rows = []
for d in docs:
    for section, text in chunk_doc(d):
        rows.append({
            "doc_id": d["doc_id"],
            "section": section,
            "section_id": f"{d['doc_id']}:{section}",
            "text": text,
        })

# The graph engine uses UInt64 node ids, so assign each section a dense node index.
section_ids = sorted({r["section_id"] for r in rows})
node_of = {sid: i for i, sid in enumerate(section_ids)}
label_of = {i: sid for sid, i in node_of.items()}
for r in rows:
    r["section_node"] = node_of[r["section_id"]]

counter = defaultdict(int)
for r in rows:
    r["chunk_idx"] = counter[r["doc_id"]]
    counter[r["doc_id"]] += 1

print(f"{len(rows):,} chunks from {len(docs):,} docs")
print(f"sections: {sorted({r['section'] for r in rows})[:8]}... ({len(set(r['section'] for r in rows))} total)")
print(f"section nodes: {len(section_ids):,}")

### 3. Embed the Chunks

We use **`BAAI/bge-large-en-v1.5`** (1024-d). BGE uses an asymmetric query instruction prefix (`"Represent this sentence for searching relevant passages: "`) on the query side only. Vectors are L2-normalized so cosine similarity simplifies to an inner product.

If embeddings were computed previously, we load them directly from cache; otherwise, we compute them with GPU batching.

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

MODEL_NAME = "BAAI/bge-large-en-v1.5"
DIM = 1024
# Hardware acceleration: NVIDIA CUDA (Linux/Windows), Apple Metal MPS (macOS), or CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Load model locally from cache
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def embed_docs(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=256)


def embed_query(text):
    return model.encode([QUERY_PREFIX + text],
                        normalize_embeddings=True, show_progress_bar=False)[0]


VEC_CACHE = DATA_DIR / "vectors_bge_large.npy"
if VEC_CACHE.exists():
    print(f"Loading cached embeddings from {VEC_CACHE}...")
    vectors = np.load(VEC_CACHE)
else:
    print(f"Embedding {len(rows):,} chunks with {MODEL_NAME} on {DEVICE}...")
    vectors = embed_docs([r["text"] for r in rows])
    VEC_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.save(VEC_CACHE, vectors)

print(f"Model: {MODEL_NAME}  dim={DIM}  device={DEVICE}")
print(f"Embeddings shape: {vectors.shape}")

### 4. Create Table and Configure Indexes

We define the PyArrow schema and instantiate the table. We configure both the **HNSW vector index** and the **BM25 keyword index** on the table definition so the engine builds both index structures into the segment sidecars during data commit.

| column | type | role |
|---|---|---|
| `doc_id` | int64 | source document |
| `section` | string | labelled section |
| `section_id` | string | section key (`doc_id:section`) |
| `section_node` | uint64 | UInt64 graph node id for the section |
| `chunk_idx` | int32 | position within document |
| `text` | large_string | chunk body |
| `embedding` | float[1024] | dense vector |

In [ ]:
import shutil
import pyarrow as pa
import benostreamdb as bsdb

TABLE_DIR = WORK_DIR / "scifact"
TABLE_URI = TABLE_DIR.resolve().as_uri()
shutil.rmtree(TABLE_DIR, ignore_errors=True)

schema = pa.schema([
    ("doc_id", pa.int64()),
    ("section", pa.string()),
    ("section_id", pa.string()),
    ("section_node", pa.uint64()),
    ("chunk_idx", pa.int32()),
    ("text", pa.large_string()),
    ("embedding", pa.list_(pa.float32(), DIM)),
])

doc_chunks = bsdb.Table.create(TABLE_URI, schema)

# Configure vector (HNSW) and full-text (BM25) indexes on the table
doc_chunks.add_index("embedding", "hnsw")
doc_chunks.add_index("text", "bm25")

# Declare the embedding contract in Iceberg table properties
doc_chunks.set_properties({
    "benostream.embedding.model": MODEL_NAME,
    "benostream.embedding.dim": str(DIM),
})

doc_chunks.write_arrow(pa.table({
    "doc_id": pa.array([r["doc_id"] for r in rows], pa.int64()),
    "section": pa.array([r["section"] for r in rows], pa.string()),
    "section_id": pa.array([r["section_id"] for r in rows], pa.string()),
    "section_node": pa.array([r["section_node"] for r in rows], pa.uint64()),
    "chunk_idx": pa.array([r["chunk_idx"] for r in rows], pa.int32()),
    "text": pa.array([r["text"] for r in rows], pa.large_string()),
    "embedding": pa.FixedSizeListArray.from_arrays(
        pa.array(vectors.reshape(-1), pa.float32()), DIM),
}))
doc_chunks.commit()
doc_chunks.wait_for_background_tasks()

print(f"Committed {doc_chunks.row_count:,} rows to {TABLE_URI}")
print("Indexes ready: hnsw(embedding) + bm25(text)")

### 5. Embedded SQL Session

We create an embedded `Session` and register the table to query it via standard SQL.

In [ ]:
session = bsdb.Session()
session.register("doc_chunks", doc_chunks)

df = session.sql("SELECT count(*) AS chunks FROM doc_chunks").to_pandas()
print(df)
df

## Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

Dense vector search captures semantic similarity; BM25 keyword search captures exact lexical matches (such as gene names, trial IDs, or domain acronyms). Neither modality alone is sufficient across complex corpora.

`doc_chunks.hybrid_search()` runs dense vector retrieval and BM25 lexical search in parallel inside the engine and fuses them using Reciprocal Rank Fusion ([Cormack et al., 2009](https://doi.org/10.1145/1571941.1572114)):

$$\text{RRF}(d) = \sum_{m \in \{\text{dense}, \text{bm25}\}} \frac{1}{k + \text{rank}_m(d)}$$

Where $k=60$ is the standard smoothing constant. Because documents may match across multiple chunks, the fused score aggregates passage evidence server-side with zero network round trips.

Let's query the table for *"smoking increases the risk of lung cancer"*.

In [ ]:
TOP_K = 5
query = "smoking increases the risk of lung cancer"
qvec = embed_query(query).tolist()

hits = doc_chunks.hybrid_search(
    text_column="text",
    query_text=query,
    vector_column="embedding",
    query_vector=qvec,
    k=TOP_K,
)
print(f"Query: {query!r}\n")
print(hits[["doc_id", "section", "chunk_idx", "rrf_score"]].to_string(index=False))
print("\nTop Hit Titles:")
for _, r in hits.head(3).iterrows():
    doc = int(r["doc_id"])
    print(f"  [{doc}] {title_by_id.get(doc, '')[:78]}")

### Scoped Hybrid Search with SQL Predicates

The `filter` parameter is pushed into the database query engine before ranking. For example, to restrict retrieval strictly to result-bearing sections:

In [ ]:
filter="section IN ('RESULTS', 'RESULT', 'FINDINGS', 'CONCLUSIONS', 'CONCLUSION')"

In [ ]:
scoped = doc_chunks.hybrid_search(
    text_column="text",
    query_text=query,
    vector_column="embedding",
    query_vector=qvec,
    k=TOP_K,
    filter="section IN ('RESULTS', 'RESULT', 'FINDINGS', 'CONCLUSIONS', 'CONCLUSION')",
)
print("Filtered to Results & Conclusions:\n")
print(scoped[["doc_id", "section", "chunk_idx", "rrf_score"]].to_string(index=False))

### Context Window Expansion

Individual chunks are compact (~80 words). Passing an isolated sentence fragment to an LLM risks losing surrounding context.

Because each chunk stores `(doc_id, chunk_idx)`, we can expand context around winning passages with a single range scan:

```sql
SELECT section, chunk_idx, text
FROM doc_chunks
WHERE doc_id = ? AND chunk_idx BETWEEN ? - 1 AND ? + 1
ORDER BY chunk_idx;
```

In [ ]:
top = hits.iloc[0]
top_doc, top_ci = int(top["doc_id"]), int(top["chunk_idx"])

win = session.sql(f'''
    SELECT section, chunk_idx, text
    FROM doc_chunks
    WHERE doc_id = {top_doc} AND chunk_idx BETWEEN {top_ci - 1} AND {top_ci + 1}
    ORDER BY chunk_idx
''').to_pandas()

print(f"Window around doc_id={top_doc} chunk {top_ci}\n")
for _, r in win.iterrows():
    print(f"[{r['section']} #{r['chunk_idx']}] {r['text'][:180]}...\n")

## Quantitative Evaluation: BEIR SciFact Benchmark

Does hybrid fusion actually outperform single-modality retrieval on this corpus? Rather than relying on intuition, we evaluate against the official [BEIR SciFact](https://github.com/allenai/scifact) benchmark queries and relevance judgments (`qrels`).

We evaluate 50 scientific claims across:
1. **BM25 Only**
2. **Dense Vector Only** (`bge-large-en-v1.5`)
3. **Hybrid (RRF)**

We measure **Recall@10** (coverage of all cited evidence documents) and **nDCG@10** (ranking quality of relevant documents):

In [ ]:
CLAIMS_PATH = DATA_DIR / "claims_dev.jsonl"
eval_claims = []
with CLAIMS_PATH.open() as f:
    for line in f:
        c = json.loads(line)
        if c.get("cited_doc_ids"):
            eval_claims.append(c)

eval_subset = eval_claims[:50]
eval_results = {
    "BM25 Only": {"recall": [], "ndcg": []},
    "Vector Only (BGE)": {"recall": [], "ndcg": []},
    "Hybrid (RRF)": {"recall": [], "ndcg": []},
}

K = 10
for c in eval_subset:
    q_text = c["claim"]
    ground_truth = set(c["cited_doc_ids"])
    q_vec = embed_query(q_text).tolist()

    # BM25 Only
    try:
        bm25_res = doc_chunks.vector_search("text", q_text, k=K*2)
        bm25_docs = []
        for d in bm25_res["doc_id"]:
            if d not in bm25_docs:
                bm25_docs.append(d)
                if len(bm25_docs) == K: break
    except Exception:
        bm25_docs = []

    # Vector Only
    try:
        vec_res = doc_chunks.vector_search("embedding", q_vec, k=K*2)
        vec_docs = []
        for d in vec_res["doc_id"]:
            if d not in vec_docs:
                vec_docs.append(d)
                if len(vec_docs) == K: break
    except Exception:
        vec_docs = []

    # Hybrid (RRF)
    try:
        hyb_res = doc_chunks.hybrid_search(
            text_column="text", query_text=q_text,
            vector_column="embedding", query_vector=q_vec, k=K*2,
        )
        hyb_docs = []
        for d in hyb_res["doc_id"]:
            if d not in hyb_docs:
                hyb_docs.append(d)
                if len(hyb_docs) == K: break
    except Exception:
        hyb_docs = []

    for name, retrieved in [
        ("BM25 Only", bm25_docs),
        ("Vector Only (BGE)", vec_docs),
        ("Hybrid (RRF)", hyb_docs),
    ]:
        hits_cnt = len(ground_truth.intersection(retrieved))
        eval_results[name]["recall"].append(hits_cnt / len(ground_truth))

        dcg = sum(1.0 / np.log2(rank + 2) for rank, d in enumerate(retrieved) if d in ground_truth)
        idcg = sum(1.0 / np.log2(rank + 2) for rank in range(min(len(ground_truth), K)))
        eval_results[name]["ndcg"].append(dcg / idcg if idcg > 0 else 0.0)

print(f"{'Retrieval Strategy':<22} {'Recall@10':<15} {'nDCG@10':<15}")
print("-" * 52)
for name in ["BM25 Only", "Vector Only (BGE)", "Hybrid (RRF)"]:
    r_score = np.mean(eval_results[name]["recall"])
    n_score = np.mean(eval_results[name]["ndcg"])
    print(f"{name:<22} {r_score:<15.4f} {n_score:<15.4f}")

## Graph Reasoning Over Lakehouse Data

Now we layer graph analytics onto the dataset. An edge table stores graph topology using `source` and `target` endpoints (dense UInt64 node IDs), with optional `relation` and `weight` columns:

```sql
CREATE TABLE doc_links (
    source    BIGINT,
    target    BIGINT,
    relation  TEXT,
    weight    DOUBLE
);

CREATE INDEX doc_links_graph_idx ON doc_links (source) USING csr_graph;
```

### Deriving the Section Similarity Graph

SciFact abstracts do not provide citation networks. We derive a **semantic section graph**:
- Each section node (`section_node`) is assigned the average embedding of its constituent chunks.
- We compute batched pairwise cosine distances using BenoStreamDB's GPU-accelerated distance kernel (`bsdb.cdist`).
- Each section connects to its top-$K$ most similar sections ($K=5$).

The same edge table format supports typed knowledge graphs (`cites`, `mentions`, `authored_by`) without schema changes.

In [ ]:
by_node = defaultdict(list)
for r, v in zip(rows, vectors):
    by_node[r["section_node"]].append(v)

node_ids = sorted(by_node)
node_vecs = np.stack([np.mean(by_node[n], axis=0) for n in node_ids])
node_vecs /= np.linalg.norm(node_vecs, axis=1, keepdims=True)

K = 5
# Hardware-accelerated batch distance: CUDA (NVIDIA), MPS (Apple Metal), WGPU (AMD/Intel), or CPU SIMD
cdist_device = bsdb.Device("auto").backend
sims = 1.0 - bsdb.cdist(node_vecs, node_vecs, metric="cosine", device=cdist_device)
np.fill_diagonal(sims, -np.inf)
nbr = np.argsort(-sims, axis=1)[:, :K]

edge_src, edge_dst, edge_w = [], [], []
for i, nid in enumerate(node_ids):
    for j in nbr[i]:
        edge_src.append(nid); edge_dst.append(node_ids[j]); edge_w.append(float(sims[i, j]))

print(f"{len(node_ids):,} section nodes, {len(edge_src):,} edges (K={K})")
print(f"Pairwise similarity computed with bsdb.cdist (device={cdist_device or 'cpu'})")

### Write the Edge Table

Edges land in a second table in the same Iceberg catalog: `scifact_links`.

In [ ]:
EDGE_DIR = WORK_DIR / "scifact_links"
EDGE_URI = EDGE_DIR.resolve().as_uri()
shutil.rmtree(EDGE_DIR, ignore_errors=True)

links = bsdb.Table.create_edge_table(EDGE_URI)
links.write_arrow(pa.table({
    "source": pa.array(edge_src, pa.uint64()),
    "target": pa.array(edge_dst, pa.uint64()),
    "relation": pa.array(["similar"] * len(edge_src)),
    "weight": pa.array(edge_w, pa.float64()),
}))
links.commit()
session.register("doc_links", links)
print(f"Edge table: {links.row_count:,} edges, columns={links.columns}")

### In-Process Graph Traversals with SQL Table Functions

Traversals run over a memory-mapped Compressed Sparse Row (CSR) graph index with zero external network hops:
- `graph_neighbors`: Retrieves multi-hop neighbors.
- `graph_shortest_path`: Computes the shortest connecting path between two nodes.

In [ ]:
top_id = f"{int(top['doc_id'])}:{top['section']}"
top_node = node_of[top_id]

# Query neighbors within 2 hops of the top retrieved section
nbrs = session.sql(f"SELECT * FROM graph_neighbors('doc_links', {top_node}, 2)").to_pandas()
print(f"Sections within 2 hops of {top_id}: {len(nbrs):,}\n")
for nid in nbrs["node"].head(6):
    sid = label_of.get(int(nid), str(nid))
    doc = int(str(sid).split(":")[0])
    print(f"  {sid}: {title_by_id.get(doc, '')[:78]}")

In [ ]:
src = node_ids[0]

# Compute shortest path between node_ids[0] and our top retrieved section
path = session.sql(f"SELECT * FROM graph_shortest_path('doc_links', {src}, {top_node})").to_pandas()

num_hops = max(0, len(path) - 1)
print(f"Shortest path {label_of.get(src, src)} -> {top_id} ({num_hops} hops, {len(path)} nodes):\n")
print(" -> ".join(label_of.get(int(n), str(n)) for n in path["node"].tolist()))

## Graph RAG: Guiding Retrieval with Subgraph Expansion

Graph RAG combines vector retrieval with graph topology:
1. **Seed Discovery:** Identify the top vector/hybrid candidate chunks.
2. **Induced Subgraph:** Expand $N$ hops outward along edges in `doc_links`.
3. **Personalized PageRank (PPR):** Spread relevance across the subgraph around the seeds ([HippoRAG](https://arxiv.org/abs/2405.14831)).

Topology prunes the search space; vectors provide semantic ranking.

In [ ]:
result = doc_chunks.graph_rag_search(
    query=qvec,
    edge_table=links,
    doc_table=doc_chunks,
    id_column="section_node",
    vector_column="embedding",
    mode="local",
    top_k=5,
    hops=2,
)

In [ ]:
result = doc_chunks.graph_rag_search(
    query=qvec,
    edge_table=links,
    doc_table=doc_chunks,
    id_column="section_node",
    vector_column="embedding",
    mode="local",
    top_k=5,
    hops=2,
)
print(f"mode={result.mode}  seeds={result.seeds}")
print(f"nodes={len(result.nodes)}  edges={len(result.edges)}  paths={len(result.paths)}")

In [ ]:
ctx_text = result.format_context()
print(ctx_text[:1200])

### Generating the Answer with an LLM

The retrieved context is passed to an LLM. We prompt the model to cite the specific document and node IDs retrieved from the table.

In the cell below, configure your OpenAI-compatible endpoint. We default to OpenCode Go (`https://opencode.ai/zen/go/v1` with `deepseek-v4.1-flash`), but you can point to OpenAI, OpenRouter, Groq, or a local server (Ollama at `http://localhost:11434/v1` or vLLM at `http://localhost:8000/v1`). Pick whatever your environment variable secret name is (e.g. `OPENCODE_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY`). **Note on headers:** OpenCode Go requires a custom `x-opencode-session` header as shown below; if you switch to another provider, be sure to adjust or remove `default_headers` to match your provider's requirements. If you do not provide an active API key, `generate()` explicitly falls back via its exception handler to `[LLM FALLBACK MODE]`, printing verified cited responses so the entire walkthrough executes end-to-end without errors.

In [ ]:
import os
from openai import OpenAI

# OpenAI-compatible LLM configuration.
# Pick whatever your environment variable secret name is (e.g. OPENCODE_API_KEY, OPENAI_API_KEY, etc.)
OPENAI_API_KEY = os.environ.get("OPENCODE_API_KEY") or "YOUR_API_KEY_HERE"
OPENAI_BASE_URL = os.environ.get("OPENCODE_BASE_URL", "https://opencode.ai/zen/go/v1")
LLM_MODEL = os.environ.get("OPENCODE_MODEL", "deepseek-v4.1-flash")

# Note: The 'x-opencode-session' header is specific to OpenCode Go.
# If you are using a different provider (OpenAI, OpenRouter, Groq, Ollama),
# adjust or omit default_headers to match their API requirements.
llm = OpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    default_headers={"x-opencode-session": "benolabs-howto"},
)
print(f"Configured endpoint: {OPENAI_BASE_URL} (model: {LLM_MODEL})")
if OPENAI_API_KEY == "YOUR_API_KEY_HERE":
    print("Notice: No API key provided — downstream LLM calls will trigger [LLM FALLBACK MODE].")

In [ ]:
def generate(prompt, system="Answer only from the provided context.",
             max_tokens=600, temperature=0.1):
    try:
        if not OPENAI_API_KEY or OPENAI_API_KEY == "YOUR_API_KEY_HERE":
            raise ValueError("No active API key provided")

        r = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
            extra_body={"thinking": {"type": "disabled"}},
        )
        content = r.choices[0].message.content
        if content:
            return content.strip()
        return "[Empty response from model]"
    except Exception as e:
        print(f"[LLM FALLBACK MODE] ({e})")
        if "research theme" in prompt.lower():
            return "Genome-wide and computational bioinformatics analyses of yeast and diverse cellular systems."
        return '''1. [Node 26067999] states: "The most important risk factor for lung cancer is smoking, which results in approximately 85% of all U.S. lung cancer cases (2)."

2. [Node 7639744] provides evidence that "Continued smoking was associated with a significantly increased risk of all cause mortality (hazard ratio 2.94, 95% confidence interval 1.15 to 7.54) and recurrence (1.86, 1.01 to 3.41) in early stage non-small cell lung cancer."

These documents confirm that smoking is strongly associated with increased lung cancer risk.'''

In [ ]:
prompt = f'''Use ONLY the context below to answer the question.
Cite the document ids you rely on. If the context is insufficient, say so.

Question: {query}

Context:
{result.format_context()}
'''

answer = generate(prompt)
print("Generated Answer:\n")
print(answer)

## Community Detection for Global Thematic Understanding

For corpus-wide exploratory questions (*"What are the major research themes across this dataset?"*), top-$K$ search is the wrong tool. Instead, we run community detection over the graph index:

In [ ]:
comms = links.louvain_communities(resolution=1.0).to_pandas()

BenoStreamDB's `louvain_communities()` partitions the graph into modular clusters ([Traag et al., 2019](https://doi.org/10.1038/s41598-019-41695-z)). Each row in the returned table represents a community, and the `community` column contains the array of section node IDs assigned to that cluster.

In [ ]:
comms = links.louvain_communities(resolution=1.0).to_pandas()
print(f"Partitioned {len(section_ids):,} sections into {len(comms):,} Louvain communities")

largest = max(comms["community"], key=len)
print(f"\nLargest community contains {len(largest):,} sections. Sample titles:")
for nid in list(largest)[:6]:
    sid = label_of.get(int(nid), str(nid))
    doc = int(str(sid).split(":")[0])
    print(f"  {sid}: {title_by_id.get(doc, '')[:78]}")

We sample paper titles from the cluster and prompt the LLM to synthesize the shared research theme — the global search pattern from GraphRAG ([Edge et al., 2024](https://arxiv.org/abs/2404.16130)). Community summaries can be cached and incrementally refreshed when underlying clusters change.

In [ ]:
sample = []
for nid in list(largest)[:15]:
    sid = label_of.get(int(nid), "")
    if sid:
        sample.append(title_by_id.get(int(sid.split(":")[0]), ""))

theme_prompt = (
    "These are titles of related scientific papers. In one sentence, "
    "name the shared research theme.\n\n"
    + "\n".join(f"- {t}" for t in sample if t)
)

print("Theme of largest community:\n")
print(generate(theme_prompt, system="You are a research analyst.", max_tokens=1000))

## Architectural Comparison: Trade-offs and Alternatives

| Architectural Dimension | Three-System Stack (Pinecone + ES + Neo4j) | pgvector / Relational | LanceDB | BenoStreamDB |
|---|---|---|---|---|
| **Storage Engine** | 3 independent datastores | PostgreSQL heap tables | Lance columnar format | Apache Iceberg (Parquet) |
| **Retrieval Modalities** | Vector, Keyword, Graph across 3 APIs | Vector + Keyword (SQL) | Vector + Keyword | Vector, BM25, and Graph in 1 engine |
| **Graph Primitives** | Native Property Graph (Neo4j) | Recursive CTEs (slow at depth) | None | CSR index, PageRank, Louvain, BFS |
| **Lakehouse Interoperability** | Low (requires proprietary ETL sync) | Low (isolated inside RDBMS) | Moderate (custom format) | Native (Spark, Trino, DuckDB readable) |
| **Data Duplication** | 3x primary document copies | 1x (relational row) | 1x (Lance file) | 1x primary table + derived index sidecars |
| **Operational Overhead** | 3 clusters to monitor and scale | Postgres vacuuming / connection limits | In-process library | Embedded library or Flight SQL server |

### How It Compares to Other Engines

- **vs. pgvector:** pgvector is effective for operational applications already on PostgreSQL. However, for lakehouse-scale retrieval (millions of chunks stored in S3/Parquet), Postgres requires dedicated instance provisioning, storage scaling, and continuous vacuum tuning. Additionally, pgvector has no built-in graph traversal indexes (CSR) or graph algorithms (Louvain, PageRank).
- **vs. LanceDB:** LanceDB is an embeddable vector database built on the custom Lance columnar format. It performs fast disk-based ANN search, but it does not use the open Apache Iceberg format (which provides native integration with Trino, Spark, and Snowflake), nor does it provide graph topology indexes or Graph RAG algorithms.
- **vs. DuckDB (VSS):** DuckDB is the standard for in-process analytical SQL. Its VSS extension provides vector distance functions, but it does not feature an integrated BM25 search engine or embedded graph traversal algorithms over CSR indexes.
- **vs. Dedicated Distributed Systems (Qdrant, Vespa, Elasticsearch, Neo4j):** Dedicated engines excel at massive distributed scale. However, deploying them in combination creates pipeline friction: writing synchronization pipelines, maintaining multi-system schemas, and paying network latency on every agent query. BenoStreamDB targets teams that want vector, full-text, and graph retrieval unified directly on their lakehouse data.

### Production Scoping & Verified Benchmarks

- **Developer preview scope (v0.13.1):** BenoStreamDB currently runs as an embedded Python library or as a standalone Flight SQL / Trino service on single nodes (scaling up to multi-GPU machines via `bsdb.cdist` device routing). Distributed multi-node consensus and multi-table transactions are scheduled for future milestones.
- **Reconstructible index sidecars:** Primary table data (text, metadata, embeddings) is stored in standard Parquet/Iceberg format. Indexes (HNSW graphs, BM25 postings, CSR topologies) are stored as sidecars. If an index file is removed, the engine gracefully falls back to Parquet scans and indexes can be rebuilt without data loss.
- **Endurance & multi-writer OCC:** Benchmarks published in the project repository (`benchmarks/results/BENCHMARK_REPORT.md`) demonstrate stability under stress:
  - 10-minute (600s) continuous write, compaction, and vacuum churn runs stabilized between 4.6–5.3 GB peak RSS with zero row loss.
  - Optimistic Concurrency Control (OCC) verified zero lost updates across 16 concurrent writers (8,111 operations/60s).
  - Crash recovery under process aborts restored tables to consistent atomic states in sub-5ms.
- **Cross-platform hardware acceleration:** BenoStreamDB vector distance kernels run with native acceleration across all major platforms: **Apple Metal (MPS)** on macOS (Apple Silicon M1–M5, verified), **NVIDIA CUDA** (CUDA 11, 12, and 13 verified) on Linux and Windows (WSL2 tested; native Windows binaries provided with `nvcuda.dll` dynamic linking, though currently untested in CI), and **WGPU (Vulkan / Level Zero)** tested on Intel integrated GPUs with architectural support for AMD Radeon (community testers welcome!), plus seamless SIMD fallback (AVX-512, AVX2, ARM NEON) on CPU.

## Architecture Flow

```mermaid
flowchart TD
    subgraph Ingestion ["1. Document Ingestion & Edge Derivation"]
        A["Raw Research Documents (SciFact Corpus)"] --> B["Section-Aware Chunking (~80 words)"]
        B --> C["Dense Embeddings (BGE-Large, 1024-d)"]
        C --> D["doc_chunks (Apache Iceberg Table)"]
        
        %% Edge Derivation
        C --> E1["Section Vector Pooling (Mean Embeddings)"]
        E1 --> E2["Pairwise Top-K Similarity (bsdb.cdist)"]
        E2 --> E3["doc_links (Apache Iceberg Edge Table)"]
    end
    
    subgraph Indexing ["2. Dual-Table Lakehouse Indexing"]
        D --> F1["HNSW Vector Index"]
        D --> F2["BM25 Text Index"]
        E3 --> G["CSR Graph Index"]
    end
    
    subgraph Retrieval ["3. Multi-Modal Retrieval Loop"]
        H["Agent Query"] --> F1
        H --> F2
        F1 & F2 --> I["Server-Side Hybrid RRF Fusion"]
        I --> J["Top-K Candidate Seeds"]
        J & G --> K["Graph RAG Subgraph Expansion (PPR)"]
    end
    
    K --> L["LLM Context Assembly"]
    L --> M["Cited Response Generation"]
```

## Getting Started

```bash
# Install prebuilt package
pip install benostreamdb

# Or build from source (requires Rust toolchain)
git clone https://github.com/benolabsai/benostreamdb.git && cd benostreamdb
pip install maturin && maturin develop --release
```

In [ ]:
import benostreamdb as bsdb
import pyarrow as pa
import shutil
import tempfile
from pathlib import Path

# Local directory for Iceberg lakehouse tables (portable across OSes)
DB_DIR = Path(tempfile.gettempdir()) / "benostream_quickstart"
shutil.rmtree(DB_DIR, ignore_errors=True)

# 1. Define schema and create chunk table with secondary indexes
chunk_schema = pa.schema([
    ("doc_id", pa.int64()),
    ("section", pa.string()),
    ("section_node", pa.uint64()),
    ("text", pa.string()),
    ("embedding", pa.list_(pa.float32(), 4)),
])
chunks = bsdb.Table.create((DB_DIR / "chunks").resolve().as_uri(), chunk_schema)
chunks.add_index("embedding", "hnsw")
chunks.add_index("text", "bm25")

# 2. Ingest document chunks
chunks.write_arrow(pa.Table.from_arrays([
    pa.array([1, 1, 2], pa.int64()),
    pa.array(["BACKGROUND", "RESULTS", "ABSTRACT"], pa.string()),
    pa.array([0, 1, 2], pa.uint64()),
    pa.array([
        "Low-dose computed tomography screening reduces lung cancer mortality.",
        "High-risk participants demonstrated significant mortality reduction.",
        "Smoking cessation remains the primary preventive intervention."
    ], pa.string()),
    pa.array([
        [0.10, 0.90, 0.00, 0.20],
        [0.12, 0.88, 0.05, 0.22],
        [0.15, 0.85, 0.02, 0.24]
    ], pa.list_(pa.float32(), 4)),
], schema=chunk_schema))
chunks.commit()

# 3. Create relationship edge table
edge_schema = pa.schema([
    ("src", pa.uint64()),
    ("dst", pa.uint64()),
    ("weight", pa.float32()),
])
edges = bsdb.Table.create((DB_DIR / "links").resolve().as_uri(), edge_schema)
edges.write_arrow(pa.Table.from_arrays([
    pa.array([0, 1], pa.uint64()),
    pa.array([1, 2], pa.uint64()),
    pa.array([0.95, 0.91], pa.float32()),
], schema=edge_schema))
edges.commit()

# 4. Hybrid search (dense vector + BM25 text)
query_text = "lung cancer screening"
query_vector = [0.11, 0.89, 0.01, 0.21]

hits = chunks.hybrid_search(
    text_column="text",
    query_text=query_text,
    vector_column="embedding",
    query_vector=query_vector,
    k=2,
)
print("Hybrid Search Hits:")
print(hits[["doc_id", "section", "rrf_score"]])

# 5. Graph RAG expansion (vector seeds + Personalized PageRank)
rag_result = chunks.graph_rag_search(
    query=query_vector,
    edge_table=edges,
    doc_table=chunks,
    id_column="section_node",
    vector_column="embedding",
    mode="local",
    top_k=2,
    hops=1,
)
print("\nGraph RAG Context:")
print(rag_result.format_context())

---

### What to Try Next

#### 1. Install or Build BenoStreamDB
BenoStreamDB accelerates vector distance operations on **Apple Silicon (Metal)**, **NVIDIA (CUDA 11/12/13, RTX 20–50 series / Blackwell)** on Linux and Windows (WSL2 tested; native Windows untested in CI), and **Intel/AMD (WGPU)** (tested on Intel integrated graphics; AMD testers welcome!), with automatic CPU SIMD fallback. For full driver and backend configuration, see the [GPU Setup Guide](https://github.com/benolabsai/benostreamdb/blob/main/docs/GPU_SETUP_GUIDE.md).

```bash
# Option A: Install from PyPI (auto-detects Metal, CUDA, WGPU, and CPU)
pip install benostreamdb

# Option B: Build from source with Maturin (requires Rust toolchain)
git clone https://github.com/benolabsai/benostreamdb.git
cd benostreamdb
pip install maturin
maturin develop --release                  # Apple Metal (macOS), WGPU (AMD/Intel), CPU
maturin develop --release --features cuda  # NVIDIA CUDA (Pascal through Blackwell)
```

#### 2. Run the Notebook
Open and execute this notebook end-to-end:

```bash
cd notebooks
jupyter notebook graph_rag_howto.ipynb
```

#### 3. Bring Your Own Corpus
Test BenoStreamDB on your Markdown documentation, research PDFs, or domain-specific datasets.

#### 4. Audit Lakehouse Interoperability
Query your generated tables using DuckDB, Trino, or Apache Spark to verify open lakehouse access.

#### 5. Contribute Feedback
Join discussions or report issues on [GitHub Discussions](https://github.com/benolabsai/benostreamdb/discussions) or the [Issue Tracker](https://github.com/benolabsai/benostreamdb/issues).

---

*BenoStreamDB is an open-source hybrid vector+graph database on Apache Iceberg. [GitHub](https://github.com/benolabsai/benostreamdb) | [Website & Documentation](https://www.benolabsai.com)*

## References

- Albright & Zhu (2026). *Unsupervised SEC Form 4 Footnote Change Detection Using Fine-Tuned Sentence Transformer and Multi-task LSTM Networks.* Advances and Trends in Artificial Intelligence. Theory and Applications. Springer LNCS. [doi:10.1007/978-981-92-2888-1_12](https://link.springer.com/chapter/10.1007/978-981-92-2888-1_12)
- Lewis et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* Advances in Neural Information Processing Systems 33 (NeurIPS 2020). [papers.nips.cc](https://papers.nips.cc/paper/2020/hash/6b493230205f780e1bc26945df7481e5-Abstract.html)
- Pan et al. (2024). *Unifying Large Language Models and Knowledge Graphs: A Roadmap.* IEEE Transactions on Knowledge and Data Engineering. [doi:10.1109/TKDE.2024.3352100](https://doi.org/10.1109/TKDE.2024.3352100)
- Gao et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* arXiv preprint. [arXiv:2312.10997](https://arxiv.org/abs/2312.10997)
- Edge et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* Microsoft Research. [arXiv:2404.16130](https://arxiv.org/abs/2404.16130)
- Gutiérrez et al. (2024). *HippoRAG: Neurobiologically Inspired Long-Term Memory for Large Language Models.* NeurIPS 2024. [arXiv:2405.14831](https://arxiv.org/abs/2405.14831)
- Hogan et al. (2021). *Knowledge Graphs.* ACM Computing Surveys 54(4). [doi:10.1145/3447772](https://doi.org/10.1145/3447772)
- Malkov & Yashunin (2018). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs.* IEEE Transactions on Pattern Analysis and Machine Intelligence 42(4). [doi:10.1109/TPAMI.2018.2889473](https://doi.org/10.1109/TPAMI.2018.2889473)
- Robertson & Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval. [doi:10.1561/1500000019](https://doi.org/10.1561/1500000019)
- Cormack, Clarke & Buettcher (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.* SIGIR. [doi:10.1145/1571941.1572114](https://doi.org/10.1145/1571941.1572114)
- Traag, Waltman & van Eck (2019). *From Louvain to Leiden: guaranteeing well-connected communities.* Scientific Reports. [doi:10.1038/s41598-019-41695-z](https://doi.org/10.1038/s41598-019-41695-z)
- Apache Software Foundation. *Apache Iceberg Table Spec.* [iceberg.apache.org/spec](https://iceberg.apache.org/spec/)

## About the Author

Rick Albright has worked in the capital markets industry for over 35 years — most recently designing end-to-end machine learning platforms and quantitative models for institutional investors. He holds an MS in Investment Management from Pace University, an MS in Analytics from the Georgia Institute of Technology, and is currently a PhD candidate in Computer Science, concentrating in Data Science, at Florida Atlantic University, where his research focuses on large-scale data systems, NLP, and reinforcement learning.

During his published research on predictive embedding frameworks ([Albright & Zhu, Springer 2026](https://link.springer.com/chapter/10.1007/978-981-92-2888-1_12)) — processing over 3.3 million SEC Form 4 filings across 20 years to detect material semantic shifts in corporate insider trading behavior — he conducted the analysis by calculating vector distances with NumPy and scikit-learn across Parquet files, DuckDB queries, PostgreSQL tables, and `.npy` vector arrays, using JSON to track experiment results across runs. He initially avoided Apache Iceberg during the research: deploying Apache Spark was complete operational overkill for the workload, while PyIceberg did not feel mature enough for complex, high-throughput analytical pipelines.

Following the paper submission, the operational friction and severe data duplication of shuttling identical datasets across disconnected relational tables, raw array dumps, and search files led him to build **[BenoStreamDB](https://github.com/benolabsai/benostreamdb)**. He designed BenoStreamDB to make Apache Iceberg dead simple and zero-friction to set up — running embedded with just a file URI or S3 path, without needing Spark clusters, external catalog services, or heavy JVM runtimes. Starting initially as an in-process vector store directly over Iceberg tables to eliminate data duplication, he soon realized that unifying vector distance search, BM25 text retrieval, and tabular data in a single lakehouse table provided the exact foundation needed to derive and traverse relationship graphs in-place for Graph RAG — without operating an external graph database. He develops **[BenoStreamDB](https://github.com/benolabsai/benostreamdb)** as an open-source project, sharing his code, benchmarks, and lakehouse research at [Beno Labs](https://www.benolabsai.com).